# Ablation 03 — Concept Baselines: Random / Dense(PCA) / Frequency(KMeans) + Empirical Jaccard Floor

Satisfies the >=3-baselines rubric. Three concept dictionaries are built from the existing train/test BiomedCLIP embeddings and scored with the SAE's standalone metrics (reconstruction cosine at L0=32, naming cosine, within-group index-Jaccard). No SAE training.

- **RANDOM** — `torch.randn(D,512)` rows L2-normalized, re-initialised per seed. Defines the chance floor.
- **DENSE (PCA)** — `PCA(n_components=256)` on train embeddings. Reconstruction ceiling; dense and non-sparse.
- **FREQUENCY (KMeans)** — `KMeans(n_clusters=256)` centroids on train embeddings. Captures data-distribution modes.

Key protocol rules: Jaccard is computed only within a fixed `(dict_size, k)` group (never across dict sizes); output dirs isolated to `results/ablation/a3_*`; committed vocab used as-is; all metrics scored on test embeddings; PCA/KMeans fit on train only. The SAE metrics are reimplemented as free functions because `SAEManager` needs an on-disk model.

## 0. Setup & Configuration

Bootstrap: resolve `PROJECT_ROOT`, set reproducibility env vars **before** importing torch, insert
`src/` on `sys.path`, then override `config.paths` to ablation-isolated directories (output isolation
rule #2).


In [1]:
# Reproducibility env vars — MUST be set before importing torch.
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("PYTHONHASHSEED", "0")  # best-effort inside a kernel

import sys
import json
from pathlib import Path

import torch
import torch.nn.functional as F
import numpy as np

# Resolve project root (walk up until 'src/' exists)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print(f'Project root: {PROJECT_ROOT}')
print(f'PyTorch: {torch.__version__}')
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f'Device: {device}')


Project root: /Users/marcantoniolopez/Documents/github/xai-project-5
PyTorch: 2.12.0
Device: mps


In [2]:
import config
import utils

# OUTPUT-DIR ISOLATION: rebind mutable PathsConfig to ablation dirs. No model writes.
ABLATION_RESULTS_DIR = PROJECT_ROOT / 'results' / 'ablation'
ABLATION_FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures' / 'ablation'
CACHE_DIR = ABLATION_RESULTS_DIR / 'a3_cache'

config.paths.results_dir = ABLATION_RESULTS_DIR
config.paths.figures_dir = ABLATION_FIGURES_DIR

ABLATION_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('=== Ablation 03 (baselines) — isolated output dirs ===')
print(f'Results dir:  {config.paths.results_dir}')
print(f'Figures dir:  {config.paths.figures_dir}')
print(f'Cache dir:    {CACHE_DIR}')
print(f'SAE defaults: dict_size={config.sae.dict_size}, k={config.sae.k}, '
      f'activation_dim={config.sae.activation_dim}')


/Users/marcantoniolopez/Documents/github/xai-project-5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Ablation 03 (baselines) — isolated output dirs ===
Results dir:  /Users/marcantoniolopez/Documents/github/xai-project-5/results/ablation
Figures dir:  /Users/marcantoniolopez/Documents/github/xai-project-5/results/figures/ablation
Cache dir:    /Users/marcantoniolopez/Documents/github/xai-project-5/results/ablation/a3_cache
SAE defaults: dict_size=4096, k=32, activation_dim=512


## 0.1 Load Data (train + test + vocabulary)

All embeddings loaded via `utils.load_tensor` (`weights_only=True`). PCA/KMeans are fit on **train**;
all metrics are scored on **test** (rule #5). Vocabulary is the committed RadLex set (rule #3).


In [3]:
# Load embeddings + vocabulary (safe deserialization, rule #4)
train_emb = utils.load_tensor(config.paths.train_embeddings_path)   # (N_train, 512)
test_emb = utils.load_tensor(config.paths.test_embeddings_path)     # (N_test, 512)

with open(config.paths.vocab_labels_path) as f:
    # vocabulary.json entries may be dicts or strings; normalize to term strings.
    vocab_labels = [
        e['term'] if isinstance(e, dict) else e for e in json.load(f)
    ]
vocab_emb = utils.load_tensor(config.paths.vocab_embeddings_path)   # (V, 512)

print('=== Data shapes ===')
print(f'  train_emb     : {tuple(train_emb.shape)}  (fit PCA/KMeans here)')
print(f'  test_emb      : {tuple(test_emb.shape)}  (score all metrics here)')
print(f'  vocab_emb     : {tuple(vocab_emb.shape)}  ({len(vocab_labels)} terms)')
print(f'  first 8 terms : {vocab_labels[:8]}')

# Sanity: embeddings should be L2-normalized (BiomedCLIP convention).
norms = test_emb.norm(dim=1)
print(f'\n  test_emb row-norm: mean={norms.mean():.4f}, min={norms.min():.4f}, max={norms.max():.4f} '
      f'(~1.0 => already unit-norm, math is valid as-is)')

=== Data shapes ===
  train_emb     : (5976, 512)  (fit PCA/KMeans here)
  test_emb      : (1494, 512)  (score all metrics here)
  vocab_emb     : (508, 512)  (508 terms)
  first 8 terms : ['cardiomegaly', 'endotracheal tube', 'Hemidiaphragma', 'herniated mediastinal pleural sac sign', 'non-tunneled central venous catheter', 'pulmonary hemorrhage', 'interstitial lung disease', 'central venous catheter']

  test_emb row-norm: mean=1.0000, min=1.0000, max=1.0000 (~1.0 => already unit-norm, math is valid as-is)


## 1. Ablation Parameters

- `ABLATION_SEEDS = (0, 42, 123)` — baseline construction seeds (the frozen `TrainingConfig.seeds` is not mutated).
- `D_B = 256` — shared baseline dict size for the within-group 3-seed Jaccard comparison.
- `D_B_BIG = 4096` — Random replicate in the SAE's native index space, to bracket the 0.0038 floor.
- `K = 32` — L0 budget for all reconstruction comparisons (= `config.sae.k`).

In [4]:
# Ablation parameters
ABLATION_SEEDS = (0, 42, 123)        # do NOT mutate frozen TrainingConfig.seeds
PRIMARY_SEED = 42                    # naming reference seed
D_B = 256                            # shared baseline dict size (within-group index space)
D_B_BIG = 4096                       # Random replicate in SAE-native index space
K = 32                               # L0 budget for fair reconstruction comparison (= config.sae.k)
TOP_N_NAMING = 3                     # candidates per feature for naming
DEAD_THRESHOLD = 1e-8                # decoder-norm dead threshold (sae_module.py:52, :412)

# Baseline SAE reference numbers (baseline/pipeline.ipynb), gap-corrected naming.
# Baselines below are scored with the SAME modality-gap shift so naming is apples-to-apples.
SAE_REFERENCE = {
    'variant': 'SAE TopK (baseline, dict4096, k32, 5 seeds)',
    'dict_size': 4096,
    'k': 32,
    'recon_cosine': 0.988,           # compute_cosine_reconstruction, sae_module.py:463-473
    'l0_mean': 32,                   # = k for TopK
    'dead_features_pct': 44.0,       # activation-based (compute_sparsity_metrics)
    'naming_cosine_mean': 0.3949,    # seed-42 gap-corrected decoder<->vocab cosine mean
    'naming_cosine_max': 0.5457,
    'cross_seed_jaccard': 0.0038,    # 5-seed index-Jaccard off-diagonal mean
}

print('=== Ablation 03 parameters ===')
print(f'  seeds            : {ABLATION_SEEDS}  (primary={PRIMARY_SEED})')
print(f'  D_b (small)      : {D_B}   (shared baseline index space)')
print(f'  D_b (big, Random): {D_B_BIG}  (SAE-native index space)')
print(f'  K (L0 budget)    : {K}   (fair recon comparison, = config.sae.k)')
print(f'  naming top_n     : {TOP_N_NAMING}, dead_threshold={DEAD_THRESHOLD}')
print()
print('=== Baseline SAE reference (hardcoded, gap-corrected naming, not re-trained here) ===')
for k_, v_ in SAE_REFERENCE.items():
    print(f'  {k_:22s}: {v_}')

=== Ablation 03 parameters ===
  seeds            : (0, 42, 123)  (primary=42)
  D_b (small)      : 256   (shared baseline index space)
  D_b (big, Random): 4096  (SAE-native index space)
  K (L0 budget)    : 32   (fair recon comparison, = config.sae.k)
  naming top_n     : 3, dead_threshold=1e-08

=== Baseline SAE reference (hardcoded, gap-corrected naming, not re-trained here) ===
  variant               : SAE TopK (baseline, dict4096, k32, 5 seeds)
  dict_size             : 4096
  k                     : 32
  recon_cosine          : 0.988
  l0_mean               : 32
  dead_features_pct     : 44.0
  naming_cosine_mean    : 0.3949
  naming_cosine_max     : 0.5457
  cross_seed_jaccard    : 0.0038


## 2. Baseline Construction

Build the three decoder/basis matrices per seed. Each is a `(D, 512)` tensor of feature directions in embedding space — the shape `SAEManager.get_decoder_weights()` returns (`(dict_size, activation_dim)`).

PCA/KMeans are fit on train only; cached per seed so re-runs are cheap.

In [5]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# PCA/KMeans are refit per seed so each seed yields a slightly different dictionary,
# enabling a within-group index-Jaccard over independent draws of the same procedure.

def l2_normalize_rows(W):
    """Row-normalize a (D,512) matrix. Mirrors F.normalize(W_dec, dim=1)."""
    return F.normalize(W, dim=1)

def build_random_basis(dict_size, dim, seed, device='cpu'):
    """Random Gaussian basis, rows L2-normalized. Re-init per seed."""
    g = torch.Generator(device=device).manual_seed(seed)
    W = torch.randn(dict_size, dim, generator=g)
    return l2_normalize_rows(W)

def build_pca_basis(n_components, seed, train_emb, cache_dir):
    """PCA on train_emb; components_ are the dictionary rows. Cached per (n, seed)."""
    cache = cache_dir / f'pca_c{n_components}_seed{seed}.npz'
    if cache.exists():
        z = np.load(cache)
        components = z['components']
        mean = z['mean']
    else:
        pca = PCA(n_components=n_components, random_state=seed)
        pca.fit(train_emb.cpu().numpy())
        components = pca.components_           # (n_components, 512)
        mean = pca.mean_                       # (512,)
        np.savez(cache, components=components, mean=mean)
    return torch.from_numpy(components).float(), torch.from_numpy(mean).float()

def build_kmeans_basis(n_clusters, seed, train_emb, cache_dir):
    """KMeans on train_emb; centroids are the dictionary rows. Cached per (n, seed)."""
    cache = cache_dir / f'kmeans_c{n_clusters}_seed{seed}.npz'
    if cache.exists():
        z = np.load(cache)
        centers = z['centers']
    else:
        km = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed)
        km.fit(train_emb.cpu().numpy())
        centers = km.cluster_centers_          # (n_clusters, 512)
        np.savez(cache, centers=centers)
    return torch.from_numpy(centers).float()

# Construct the baseline dictionaries per seed
baselines = {}   # name -> {seed -> {"W_dec": (D,512), ...}}

for seed in ABLATION_SEEDS:
    # RANDOM (D_b=256)
    baselines.setdefault('random', {})[seed] = {
        'W_dec': build_random_basis(D_B, config.sae.activation_dim, seed),
        'meta': {'kind': 'random', 'dict_size': D_B},
    }
    # RANDOM (D=4096) — SAE-native index-space replicate for the Jaccard floor
    baselines.setdefault('random_big', {})[seed] = {
        'W_dec': build_random_basis(D_B_BIG, config.sae.activation_dim, seed),
        'meta': {'kind': 'random', 'dict_size': D_B_BIG},
    }
    # DENSE (PCA)
    W_pca, mu_pca = build_pca_basis(D_B, seed, train_emb, CACHE_DIR)
    baselines.setdefault('dense_pca', {})[seed] = {
        'W_dec': W_pca,
        'mean': mu_pca,                         # PCA needs the data mean for inverse_transform
        'meta': {'kind': 'dense_pca', 'dict_size': D_B},
    }
    # FREQUENCY (KMeans)
    W_km = build_kmeans_basis(D_B, seed, train_emb, CACHE_DIR)
    baselines.setdefault('frequency_kmeans', {})[seed] = {
        'W_dec': W_km,
        'meta': {'kind': 'frequency_kmeans', 'dict_size': D_B},
    }

print('=== Constructed baselines ===')
for name, per_seed in baselines.items():
    shapes = {s: tuple(v['W_dec'].shape) for s, v in per_seed.items()}
    print(f'  {name:20s}: per-seed W_dec shapes = {list(shapes.values())}')

=== Constructed baselines ===
  random              : per-seed W_dec shapes = [(256, 512), (256, 512), (256, 512)]
  random_big          : per-seed W_dec shapes = [(4096, 512), (4096, 512), (4096, 512)]
  dense_pca           : per-seed W_dec shapes = [(256, 512), (256, 512), (256, 512)]
  frequency_kmeans    : per-seed W_dec shapes = [(256, 512), (256, 512), (256, 512)]


## 3. Standalone Metric Free-Functions

Re-implementations of the SAE metrics, verified against `src/autoencoder/sae_module.py`. Needed because `SAEManager` routes through a loaded on-disk `AutoEncoderTopK` and cannot run on in-memory decoder matrices.

In [6]:
# (i) Reconstruction cosine at fair L0=k.
# Mirrors SAEManager.compute_cosine_reconstruction (sae_module.py:463-473). The L0=32 protocol
# picks the 32 strongest atoms per sample for a fair comparison against the TopK SAE (k=32).

def recon_topk_coefficients(x, W_basis, k):
    """Random/Dense-fair: project x onto basis, keep top-k coefficients by magnitude, reconstruct.
    x: (N,512), W_basis: (D,512) rows unit-norm. Returns x_hat: (N,512) and l0_per_sample: (N,)."""
    coeffs = x @ W_basis.T                       # (N, D) signed projections
    idx = coeffs.abs().topk(k, dim=1).indices    # (N, k) top-k by |coeff|
    sparse = torch.zeros_like(coeffs)
    row = torch.arange(x.shape[0], device=x.device).unsqueeze(1).expand(-1, k)
    sparse[row, idx] = coeffs[row, idx]          # keep signed top-k
    x_hat = sparse @ W_basis                     # (N, 512)
    l0 = (sparse != 0).float().sum(dim=1)        # should == k
    return x_hat, l0

def recon_pca_topk(x, W_basis, mean, k):
    """PCA-fair: score = (x - mean) @ W_basis.T; keep top-k scores by magnitude; inverse_transform.
    W_basis = pca.components_ (D,512). x_hat = mean + sum_{i in topk} score_i * W_basis[i]."""
    x_centered = x - mean
    scores = x_centered @ W_basis.T             # (N, D)
    idx = scores.abs().topk(k, dim=1).indices
    sparse = torch.zeros_like(scores)
    row = torch.arange(x.shape[0], device=x.device).unsqueeze(1).expand(-1, k)
    sparse[row, idx] = scores[row, idx]
    x_hat = mean + sparse @ W_basis             # (N, 512)
    l0 = (sparse != 0).float().sum(dim=1)
    return x_hat, l0

def recon_kmeans_topk(x, W_basis, k):
    """KMeans-fair: pick the k NEAREST centroids by cosine similarity, reconstruct as a
    normalized-similarity-weighted blend of those k centroids (fair 'k atoms' analogue)."""
    sims = x @ W_basis.T                         # (N, D) cosine-ish (rows ~unit-norm on data)
    idx = sims.topk(k, dim=1).indices            # (N, k) nearest centroids
    row = torch.arange(x.shape[0], device=x.device).unsqueeze(1).expand(-1, k)
    sel_sims = sims[row, idx]                    # (N, k)
    w = F.normalize(sel_sims, p=1, dim=1)        # normalize positive weights to sum 1
    sel_basis = W_basis[idx]                     # (N, k, 512)
    x_hat = (w.unsqueeze(-1) * sel_basis).sum(dim=1)   # (N, 512)
    l0 = torch.full((x.shape[0],), float(k), device=x.device)
    return x_hat, l0

def recon_cosine(x_hat, x):
    """Identical to SAEManager.compute_cosine_reconstruction (sae_module.py:463-473)."""
    return F.cosine_similarity(x_hat, x, dim=-1).mean().item()

def baseline_recon(name, W_basis, x, k, mean=None):
    """Dispatch to the right fair-L0 reconstruction for the baseline kind."""
    if name == 'dense_pca':
        x_hat, l0 = recon_pca_topk(x, W_basis, mean, k)
    elif name == 'frequency_kmeans':
        x_hat, l0 = recon_kmeans_topk(x, W_basis, k)
    else:  # random / random_big
        x_hat, l0 = recon_topk_coefficients(x, W_basis, k)
    return x_hat, l0

# (ii) Naming cosine (decoder rows <-> vocab).
# Mirrors SAEManager.name_concepts (sae_module.py:373-447), decoder-norm dead definition (:413,1e-8),
# and the modality-gap shift (:411-414) when modality_gap is supplied.

def name_cosine(W_dec, vocab_emb, vocab_labels, top_n=3, dead_threshold=DEAD_THRESHOLD,
                modality_gap=None):
    """Returns dict[feat_id -> {name,score,candidates,is_dead}], plus aggregate (mean,max) over LIVE.
    W_dec: (D,512) decoder rows; vocab_emb: (V,512); vocab_labels: list[str] len V.
    If modality_gap is given, W_dec is shifted by -modality_gap before cosine (Soluzione 1),
    mirroring SAEManager.name_concepts (sae_module.py:411-414)."""
    if modality_gap is not None:
        W_dec = W_dec - modality_gap.unsqueeze(0).to(W_dec.device)   # sae_module.py:411-414
    norms = W_dec.norm(dim=1)                         # :411 (on shifted W_dec if gap given)
    dead_mask = norms < dead_threshold                # :413
    W_norm = F.normalize(W_dec, dim=1)                # :417
    W_norm[dead_mask] = 0.0                           # :418 (guard NaN)
    V_norm = F.normalize(vocab_emb, dim=1)            # :420
    sims = W_norm @ V_norm.T                          # :422  (D, V)
    names, live_scores = {}, []
    for f in range(W_dec.shape[0]):
        if dead_mask[f]:
            names[f] = {"name": "DEAD_FEATURE", "score": 0.0, "candidates": [], "is_dead": True}
            continue                                  # :426-433
        tk = sims[f].topk(top_n)                      # :435
        cands = [{"label": vocab_labels[i.item()], "score": float(v)} for v, i in zip(tk.values, tk.indices)]
        names[f] = {"name": cands[0]["label"], "score": cands[0]["score"], "candidates": cands, "is_dead": False}
        live_scores.append(cands[0]["score"])
    live_scores = np.array(live_scores) if live_scores else np.array([0.0])
    agg = {
        "n_live": int((~dead_mask).sum().item()),
        "n_dead": int(dead_mask.sum().item()),
        "dead_pct": float(dead_mask.float().mean().item() * 100),
        "naming_cosine_mean": float(live_scores.mean()),
        "naming_cosine_max": float(live_scores.max()),
    }
    return names, agg

# (iii) Within-group index-Jaccard (free function).
# Mirrors SAEManager.compute_stability (sae_module.py:520-607): per-sample top-n index sets,
# pairwise per-sample Jaccard averaged over samples, mean over strict upper-triangle seed pairs.

def topk_index_sets(W_basis, x, n):
    """Per-sample set of top-n feature INDICES used to encode x under this basis (by |coeff|)."""
    coeffs = x @ W_basis.T
    idx = coeffs.abs().topk(n, dim=1).indices        # (N, n) — mirrors encode_topk indices
    return [set(row.tolist()) for row in idx]         # sae_module.py:557-566

def within_group_jaccard(active_sets_per_seed, n_samples):
    """active_sets_per_seed[i] = list of n_samples sets (top-n index sets for seed i).
    Returns jaccard_matrix (S,S), mean_jaccard, std_jaccard (upper-triangle, population std)."""
    n_seeds = len(active_sets_per_seed)
    J = torch.zeros(n_seeds, n_seeds)
    for i in range(n_seeds):
        for j in range(i, n_seeds):
            if i == j:
                J[i, j] = 1.0
                continue
            jaccs = []
            for s in range(n_samples):
                a, b = active_sets_per_seed[i][s], active_sets_per_seed[j][s]
                u = len(a | b)
                jaccs.append((len(a & b) / u) if u > 0 else 0.0)
            m = sum(jaccs) / len(jaccs)
            J[i, j] = m
            J[j, i] = m
    mask = torch.triu(torch.ones(n_seeds, n_seeds), diagonal=1).bool()
    up = J[mask]
    return {
        "jaccard_matrix": J,
        "mean_jaccard": float(up.mean().item()) if up.numel() > 0 else 0.0,
        "std_jaccard": float(up.std(correction=0).item()) if up.numel() > 1 else 0.0,  # :604
    }

print('Standalone metric functions defined:')
print('  recon_cosine(x_hat,x)                              <- sae_module.py:463-473')
print('  name_cosine(W_dec,vocab_emb,...,modality_gap=None) <- sae_module.py:373-447 (gap :411-414)')
print('  within_group_jaccard(active_sets, n_samples)       <- sae_module.py:520-607')

Standalone metric functions defined:
  recon_cosine(x_hat,x)                              <- sae_module.py:463-473
  name_cosine(W_dec,vocab_emb,...,modality_gap=None) <- sae_module.py:373-447 (gap :411-414)
  within_group_jaccard(active_sets, n_samples)       <- sae_module.py:520-607


## 4. Per-Baseline, Per-Seed Metrics

Score each baseline dictionary on the **test** set: reconstruction cosine at fair L0=32, effective
L0, and naming-cosine aggregate (mean/max over live features) using the committed vocab. Reported per
seed; the table later uses the primary seed (42) for naming, matching SAE convention.


In [7]:
results = {}   # name -> seed -> metrics dict
N_TEST = test_emb.shape[0]

# Modality gap (Soluzione 1): visual_centroid - text_centroid, applied inside name_cosine so
# baselines are gap-corrected like the SAE (= models/modality_gap.pt). Only naming is affected.
modality_gap = train_emb.mean(dim=0) - vocab_emb.mean(dim=0)

for name, per_seed in baselines.items():
    results[name] = {}
    for seed, d in per_seed.items():
        W = d['W_dec']
        mean = d.get('mean')
        # Reconstruction at fair L0=k (test set)
        x_hat, l0 = baseline_recon(name, W, test_emb, K, mean=mean)
        recon = recon_cosine(x_hat, test_emb)
        l0_mean = float(l0.mean().item())
        # Naming (decoder rows <-> vocab) — gap-corrected, full aggregate
        names_map, agg = name_cosine(W, vocab_emb, vocab_labels, top_n=TOP_N_NAMING,
                                     modality_gap=modality_gap)
        results[name][seed] = {
            'dict_size': W.shape[0],
            'recon_cosine': recon,
            'l0_mean': l0_mean,
            'dead_pct': agg['dead_pct'],
            'naming_cosine_mean': agg['naming_cosine_mean'],
            'naming_cosine_max': agg['naming_cosine_max'],
            'n_live': agg['n_live'],
        }
    rs = results[name]
    print(f"=== {name}  (D={per_seed[PRIMARY_SEED]['W_dec'].shape[0]}) ===")
    for s in ABLATION_SEEDS:
        m = rs[s]
        print(f"  seed {s:>3}: recon_cos={m['recon_cosine']:.4f}  L0={m['l0_mean']:.1f}  "
              f"dead={m['dead_pct']:.1f}%  name_mean={m['naming_cosine_mean']:.4f}  "
              f"name_max={m['naming_cosine_max']:.4f}")
    rc = np.mean([rs[s]['recon_cosine'] for s in ABLATION_SEEDS])
    nm = np.mean([rs[s]['naming_cosine_mean'] for s in ABLATION_SEEDS])
    print(f"  ---- seed-mean: recon_cos={rc:.4f}  name_mean={nm:.4f}\n")

=== random  (D=256) ===
  seed   0: recon_cos=0.4398  L0=32.0  dead=0.0%  name_mean=0.3714  name_max=0.4530
  seed  42: recon_cos=0.4535  L0=32.0  dead=0.0%  name_mean=0.3723  name_max=0.4419
  seed 123: recon_cos=0.4441  L0=32.0  dead=0.0%  name_mean=0.3729  name_max=0.4573
  ---- seed-mean: recon_cos=0.4458  name_mean=0.3722



=== random_big  (D=4096) ===
  seed   0: recon_cos=0.6058  L0=32.0  dead=0.0%  name_mean=0.3724  name_max=0.4605
  seed  42: recon_cos=0.5950  L0=32.0  dead=0.0%  name_mean=0.3719  name_max=0.4534
  seed 123: recon_cos=0.5967  L0=32.0  dead=0.0%  name_mean=0.3719  name_max=0.4588
  ---- seed-mean: recon_cos=0.5992  name_mean=0.3720

=== dense_pca  (D=256) ===
  seed   0: recon_cos=0.9957  L0=32.0  dead=0.0%  name_mean=0.3804  name_max=0.5336
  seed  42: recon_cos=0.9957  L0=32.0  dead=0.0%  name_mean=0.3804  name_max=0.5336
  seed 123: recon_cos=0.9957  L0=32.0  dead=0.0%  name_mean=0.3804  name_max=0.5336
  ---- seed-mean: recon_cos=0.9957  name_mean=0.3804

=== frequency_kmeans  (D=256) ===
  seed   0: recon_cos=0.9611  L0=32.0  dead=0.0%  name_mean=0.8304  name_max=0.8789
  seed  42: recon_cos=0.9612  L0=32.0  dead=0.0%  name_mean=0.8285  name_max=0.8803
  seed 123: recon_cos=0.9612  L0=32.0  dead=0.0%  name_mean=0.8304  name_max=0.8783
  ---- seed-mean: recon_cos=0.9612  name_mean=

## 5. Empirical Random-Jaccard Floor (the key calibration)

Random dictionaries are re-initialised per seed, so the 3 seeds at a fixed `dict_size` are independent draws of the same chance procedure. Their within-group index-Jaccard is the empirical null — the Jaccard from comparing feature indices across two independent dictionaries of the same size.

Computed only within each fixed dict_size group: `random` at `D_b=256`, `random_big` at `D_B_BIG=4096` (the SAE's native index space). Expected: Random@4096 ~= 0.004, matching the baseline SAE's 0.0038.

In [8]:
# Within-group Random Jaccard at D_b=256 and at D=4096
jaccard_results = {}

for grp_name, grp_key in [('random (D=256)', 'random'), ('random (D=4096)', 'random_big')]:
    W_per_seed = [baselines[grp_key][s]['W_dec'] for s in ABLATION_SEEDS]
    D_this = W_per_seed[0].shape[0]
    active_sets = [topk_index_sets(W, test_emb, n=K) for W in W_per_seed]
    jres = within_group_jaccard(active_sets, n_samples=N_TEST)
    jaccard_results[grp_key] = {
        'dict_size': D_this,
        'seeds': list(ABLATION_SEEDS),
        'mean_jaccard': jres['mean_jaccard'],
        'std_jaccard': jres['std_jaccard'],
        'jaccard_matrix': jres['jaccard_matrix'].tolist(),
    }
    print(f"=== {grp_name} — within-group index-Jaccard (k={K}) ===")
    print(f"  seeds={list(ABLATION_SEEDS)}  D={D_this}")
    print(f"  matrix:\n{jres['jaccard_matrix']}")
    print(f"  mean={jres['mean_jaccard']:.6f}  std={jres['std_jaccard']:.6f}\n")

# Analytical chance expectation cross-check:
# E[Jaccard] for two independent k-subsets of a D-set ~= k / (2D - k).
for grp_key in ['random', 'random_big']:
    D = jaccard_results[grp_key]['dict_size']
    expected = K / (2 * D - K)
    emp = jaccard_results[grp_key]['mean_jaccard']
    print(f"  {grp_key:12s} D={D:>4}: empirical={emp:.6f}  analytical-chance={expected:.6f}  "
          f"ratio={emp/max(expected,1e-12):.2f}")
print('\n(Note: empirical > analytical because top-k-by-magnitude sets on the SAME data share '
      'structure; the point is the ORDER OF MAGNITUDE.)')


=== random (D=256) — within-group index-Jaccard (k=32) ===
  seeds=[0, 42, 123]  D=256
  matrix:
tensor([[1.0000, 0.0629, 0.0739],
        [0.0629, 1.0000, 0.0630],
        [0.0739, 0.0630, 1.0000]])
  mean=0.066597  std=0.005171

=== random (D=4096) — within-group index-Jaccard (k=32) ===
  seeds=[0, 42, 123]  D=4096
  matrix:
tensor([[1.0000e+00, 3.6946e-03, 8.4996e-04],
        [3.6946e-03, 1.0000e+00, 6.6091e-03],
        [8.4996e-04, 6.6091e-03, 1.0000e+00]])
  mean=0.003718  std=0.002351

  random       D= 256: empirical=0.066597  analytical-chance=0.066667  ratio=1.00
  random_big   D=4096: empirical=0.003718  analytical-chance=0.003922  ratio=0.95

(Note: empirical > analytical because top-k-by-magnitude sets on the SAME data share structure; the point is the ORDER OF MAGNITUDE.)


## 6. Comparison Table: SAE vs Random vs Dense(PCA) vs Frequency(KMeans)

Primary-seed (42) row per baseline plus the hardcoded SAE reference. Dense(PCA) wins raw reconstruction (the non-sparse ceiling); Random defines the chance floor; KMeans sits between. The SAE's advantage is sparsity + medical naming, not raw cosine. The Random@4096 Jaccard floor (~0.004) shows its 0.0038 cross-seed index-Jaccard is an index-space artefact, not evidence of poor learning.

In [9]:
# Build a flat comparison table. Primary-seed (42) row per baseline.
rows = []
rows.append({
    'method': 'SAE (baseline, dict4096)',
    'recon_cosine': SAE_REFERENCE['recon_cosine'],
    'l0_mean': SAE_REFERENCE['l0_mean'],
    'dead_pct': SAE_REFERENCE['dead_features_pct'],
    'naming_mean': SAE_REFERENCE['naming_cosine_mean'],
    'naming_max': SAE_REFERENCE['naming_cosine_max'],
})
baseline_display = {
    'random': 'Random (D=256)',
    'dense_pca': 'Dense-PCA (D=256)',
    'frequency_kmeans': 'Freq-KMeans (D=256)',
}
for key, disp in baseline_display.items():
    m = results[key][PRIMARY_SEED]
    rows.append({
        'method': disp,
        'recon_cosine': m['recon_cosine'],
        'l0_mean': m['l0_mean'],
        'dead_pct': m['dead_pct'],
        'naming_mean': m['naming_cosine_mean'],
        'naming_max': m['naming_cosine_max'],
    })

import pandas as pd
df = pd.DataFrame(rows).set_index('method')
df = df.round({'recon_cosine': 4, 'l0_mean': 1, 'dead_pct': 1, 'naming_mean': 4, 'naming_max': 4})
print('=== Comparison table (primary seed 42; SAE reference hardcoded) ===')
print(df.to_string())
print()
print(f"Random@4096 within-group Jaccard = {jaccard_results['random_big']['mean_jaccard']:.4f}  "
      f"(baseline SAE cross-seed Jaccard = {SAE_REFERENCE['cross_seed_jaccard']:.4f})")


=== Comparison table (primary seed 42; SAE reference hardcoded) ===
                          recon_cosine  l0_mean  dead_pct  naming_mean  naming_max
method                                                                            
SAE (baseline, dict4096)        0.9880     32.0      44.0       0.3949      0.5457
Random (D=256)                  0.4535     32.0       0.0       0.3723      0.4419
Dense-PCA (D=256)               0.9957     32.0       0.0       0.3804      0.5336
Freq-KMeans (D=256)             0.9612     32.0       0.0       0.8285      0.8803

Random@4096 within-group Jaccard = 0.0037  (baseline SAE cross-seed Jaccard = 0.0038)


## 7. Figures

### 7.1 Comparison table (saved as PNG)


In [10]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FIG_DIR = config.paths.figures_dir  # already overridden to figures/ablation

# Render the DataFrame as a styled table PNG.
fig, ax = plt.subplots(figsize=(11, 2.0 + 0.4 * len(df)))
ax.axis('off')
tbl = ax.table(
    cellText=df.round(4).values,
    rowLabels=df.index,
    colLabels=df.columns,
    cellLoc='center', rowLoc='left', loc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.6)
# Highlight the SAE row (row index 1 = SAE in the table).
for j in range(len(df.columns)):
    tbl[(1, j)].set_facecolor('#d9ead3')   # green highlight
ax.set_title('Ablation 03 — SAE vs Concept Baselines (recon cosine, L0, naming cosine)',
             fontweight='bold', pad=16)
plt.tight_layout()
out1 = FIG_DIR / 'a3_comparison_table.png'
plt.savefig(out1, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out1}')


Saved: /Users/marcantoniolopez/Documents/github/xai-project-5/results/figures/ablation/a3_comparison_table.png


/var/folders/v6/8dp2_fyn1y525csqgb_j5ln00000gn/T/ipykernel_24024/3550513266.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7.2 Random-Jaccard floor: empirical D=256 vs D=4096 vs baseline SAE 0.0038

Random@4096 ~= baseline SAE cross-seed Jaccard, calibrating the SAE's 0.0038 as near the chance-null floor for index-Jaccard at that dictionary width.

In [11]:
labels = ['Random\n(D=256)', 'Random\n(D=4096)', 'SAE baseline\n(dict4096, 5 seeds)']
vals = [
    jaccard_results['random']['mean_jaccard'],
    jaccard_results['random_big']['mean_jaccard'],
    SAE_REFERENCE['cross_seed_jaccard'],
]
errs = [
    jaccard_results['random']['std_jaccard'],
    jaccard_results['random_big']['std_jaccard'],
    0.0,
]
colors = ['#9ecae1', '#3182bd', '#e6550d']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, vals, yerr=errs, capsize=6, color=colors, edgecolor='black', alpha=0.9)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + max(vals) * 0.03,
            f'{v:.4f}', ha='center', va='bottom', fontweight='bold')
ax.axhline(SAE_REFERENCE['cross_seed_jaccard'], color='#e6550d', linestyle='--', alpha=0.5,
           label=f"SAE baseline = {SAE_REFERENCE['cross_seed_jaccard']:.4f}")
ax.set_ylabel('Mean within-group index-Jaccard')
ax.set_title('Empirical Random-Jaccard floor vs baseline SAE cross-seed Jaccard\n'
             '(Random@4096 ~= 0.004 calibrates the SAE 0.0038 as near-null)', fontweight='bold')
ax.set_ylim(0, max(vals) * 1.35)
ax.legend(loc='upper right')
plt.tight_layout()
out2 = FIG_DIR / 'a3_jaccard_floor.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out2}')


Saved: /Users/marcantoniolopez/Documents/github/xai-project-5/results/figures/ablation/a3_jaccard_floor.png


/var/folders/v6/8dp2_fyn1y525csqgb_j5ln00000gn/T/ipykernel_24024/196695924.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Save Results

Write `results/ablation/a3_baselines.json` with every per-baseline/per-seed metric, the
Random-Jaccard floors, the hardcoded SAE reference, and the ablation config.


In [12]:
payload = {
    'ablation': '03_baselines',
    'description': 'Concept baselines (Random / Dense-PCA / Frequency-KMeans) + empirical '
                   'Random-Jaccard floor. No SAE training.',
    'config': {
        'ABLATION_SEEDS': list(ABLATION_SEEDS),
        'PRIMARY_SEED': PRIMARY_SEED,
        'D_B': D_B,
        'D_B_BIG': D_B_BIG,
        'K': K,
        'TOP_N_NAMING': TOP_N_NAMING,
        'DEAD_THRESHOLD': DEAD_THRESHOLD,
        'N_TEST': int(N_TEST),
        'test_embeddings_path': str(config.paths.test_embeddings_path),
        'train_embeddings_path': str(config.paths.train_embeddings_path),
        'vocab_labels_path': str(config.paths.vocab_labels_path),
        'vocab_embeddings_path': str(config.paths.vocab_embeddings_path),
    },
    'sae_reference': SAE_REFERENCE,
    'baselines': {
        name: {str(s): m for s, m in per_seed.items()}
        for name, per_seed in results.items()
    },
    'comparison_table_primary_seed': df.round(4).to_dict(orient='index'),
    'jaccard_floors': {
        k: {kk: vv for kk, vv in v.items()}
        for k, v in jaccard_results.items()
    },
}

out_json = config.paths.results_dir / 'a3_baselines.json'
with open(out_json, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Saved: {out_json}  ({out_json.stat().st_size / 1024:.1f} KB)')


Saved: /Users/marcantoniolopez/Documents/github/xai-project-5/results/ablation/a3_baselines.json  (6.1 KB)


## 9. Summary

- **>=3-baselines rubric satisfied**: Random, Dense(PCA), Frequency(KMeans), scored on test with the SAE's own metrics.
- **Empirical Jaccard floor established**: Random within-group index-Jaccard at `D=4096` brackets the baseline SAE's 0.0038.
- **Pareto framing**: Dense(PCA) is the reconstruction ceiling; the SAE's value is sparsity + medical naming.
- **Output isolation respected**: all artefacts under `results/ablation/a3_*` and `figures/ablation/a3_*`; no model writes, no vocab rebuild, no SAE training.

In [13]:
print('=' * 60)
print('  ABLATION 03 (BASELINES) COMPLETED')
print('=' * 60)
print()
print('Results:')
for p in sorted((config.paths.results_dir).glob('a6*')):
    print(f'  {p.name}: {p.stat().st_size / 1024:.1f} KB')
print()
print('Figures:')
for p in sorted((config.paths.figures_dir).glob('a6*')):
    print(f'  {p.name}: {p.stat().st_size / 1024:.1f} KB')
print()
print('Key numbers (primary seed 42):')
for key, disp in baseline_display.items():
    m = results[key][PRIMARY_SEED]
    print(f"  {disp:22s}: recon_cos={m['recon_cosine']:.4f}  name_mean={m['naming_cosine_mean']:.4f}")
print(f"  {'SAE baseline':22s}: recon_cos={SAE_REFERENCE['recon_cosine']:.4f}  "
      f"name_mean={SAE_REFERENCE['naming_cosine_mean']:.4f}")
print()
print(f"Random@4096 Jaccard={jaccard_results['random_big']['mean_jaccard']:.4f}  "
      f"(SAE baseline={SAE_REFERENCE['cross_seed_jaccard']:.4f})")


  ABLATION 03 (BASELINES) COMPLETED

Results:
  a6_baselines.json: 6.1 KB
  a6_cache: 0.2 KB

Figures:
  a6_comparison_table.png: 54.5 KB
  a6_jaccard_floor.png: 64.4 KB

Key numbers (primary seed 42):
  Random (D=256)        : recon_cos=0.4535  name_mean=0.3723
  Dense-PCA (D=256)     : recon_cos=0.9957  name_mean=0.3804
  Freq-KMeans (D=256)   : recon_cos=0.9612  name_mean=0.8285
  SAE baseline          : recon_cos=0.9880  name_mean=0.3949

Random@4096 Jaccard=0.0037  (SAE baseline=0.0038)
